# v1.1 Data Staleness QA

## Scope

This notebook documents the **conclusive QA experiments** used to investigate crime snapshot staleness in the Seattle Area Public Safety Dashboard.

The goal was to answer:

1. How far can the rolling/incremental snapshot diverge from a current full source pull?
2. Are missing records primarily explained by late reporting?
3. Does the 30-day `report_date_time` overlap recover records that should be in scope?
4. What discrepancies remain after simulating the production incremental refresh?
5. What failure modes are inherent to the rolling strategy?

Only experiments that materially contributed to the final conclusions are included here. Exploratory or inconclusive checks were intentionally omitted.

## Executive summary

Observed during the QA run:

| QA result | Observed value |
|---|---:|
| Offenses missing from the stale rolling snapshot | 1,103 |
| Missing offenses reported after the rolling cutoff | 959 |
| Share explained by later reporting | 86.94% |
| Missing offenses not explained by later reporting | 144 |
| Of those 144, inside the 30-day overlap | 125 |
| Of those 144, outside the 30-day overlap | 19 |
| In-overlap missing IDs returned by a fresh rolling-style query | 125 / 125 |
| Full-source unique offenses after aligned-window simulation | 156,934 |
| Simulated rolling unique offenses | 156,919 |
| Missing after simulated incremental refresh | 19 |
| Extra after simulated incremental refresh | 4 |
| Residual missing records outside overlap | 19 / 19 |
| Residual extra records absent from current source entirely | 4 / 4 |

### Final finding

The rolling refresh logic can recover recent late-reported records, but it cannot guarantee parity with the current source because:

- records introduced or changed outside the overlap window are not revisited; and
- upstream deletions are not removed by an append/merge/deduplicate strategy.

Because dashboard correctness is the priority, these findings support using a **daily full refresh** for the retained crime window, with validation before replacing the production snapshot.


## 1. Setup

The notebook finds the repository root dynamically so it can be run from either the repo root or the `notebooks/` directory.

The production crime snapshot is loaded from `data/processed/crime`, while the current source data is fetched through the same service layer used by the application.


In [5]:
from pathlib import Path
from datetime import timedelta
import sys

import pandas as pd
from IPython.display import display


def find_repo_root(start: Path | None = None) -> Path:
    current = Path(start or Path.cwd()).resolve()

    for candidate in [current, *current.parents]:
        if (candidate / "dashboard").exists() and (candidate / "notebooks").exists():
            return candidate

    raise RuntimeError("Could not locate repository root.")


REPO_ROOT = find_repo_root()

if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from dashboard.crime_service import load_crime_dataset
from dashboard.crime_snapshot import load_crime_snapshot


CRIME_SNAPSHOT_DIR = REPO_ROOT / "data" / "processed" / "crime"

CRIME_KEY = "offense_id"
CRIME_EVENT_TIME = "offense_date"
CRIME_REFRESH_TIME = "report_date_time"

ROLLING_WINDOW_DAYS = 734
OVERLAP_DAYS = 30

print("Repository root:", REPO_ROOT)
print("Crime snapshot directory:", CRIME_SNAPSHOT_DIR)


Repository root: C:\Users\benca\code\PersonalPythonProjects\SPDCallDashboard
Crime snapshot directory: C:\Users\benca\code\PersonalPythonProjects\SPDCallDashboard\data\processed\crime


## 2. Load the rolling snapshot and a current full source pull

The full-source pull is anchored to the earliest offense date present in the rolling snapshot. This ensures the source pull contains enough history for all later aligned-window comparisons.


In [6]:
rolling_crime, crime_metadata = load_crime_snapshot(CRIME_SNAPSHOT_DIR)

for column in [CRIME_EVENT_TIME, CRIME_REFRESH_TIME]:
    rolling_crime[column] = pd.to_datetime(
        rolling_crime[column],
        errors="coerce",
    )

crime_start = (
    rolling_crime[CRIME_EVENT_TIME]
    .min()
    .date()
    .isoformat()
)

full_crime = load_crime_dataset(
    start_date=crime_start,
    page_size=5000,
    max_pages=None,
    date_column=CRIME_EVENT_TIME,
    timeout=30,
    max_retries=5,
    retry_backoff_seconds=5
)

for column in [CRIME_EVENT_TIME, CRIME_REFRESH_TIME]:
    full_crime[column] = pd.to_datetime(
        full_crime[column],
        errors="coerce",
    )

print("Rolling rows:", len(rolling_crime))
print("Full-source rows:", len(full_crime))
print("Rolling latest offense_date:", rolling_crime[CRIME_EVENT_TIME].max())
print("Full-source latest offense_date:", full_crime[CRIME_EVENT_TIME].max())


Rolling rows: 156694
Full-source rows: 159562
Rolling latest offense_date: 2026-09-21 22:44:00
Full-source latest offense_date: 2026-09-30 21:30:00


## 3. Establish the original missing-record discrepancy

For the first comparison, the full source is restricted to the same `offense_date` bounds as the stale rolling snapshot. This prevents newer source records from being counted as missing simply because they occurred after the rolling snapshot's latest date.


In [7]:
rolling_min = rolling_crime[CRIME_EVENT_TIME].min()
rolling_max = rolling_crime[CRIME_EVENT_TIME].max()

rolling_crime_compare = rolling_crime[
    rolling_crime[CRIME_EVENT_TIME].between(
        rolling_min,
        rolling_max,
    )
].copy()

full_crime_compare = full_crime[
    full_crime[CRIME_EVENT_TIME].between(
        rolling_min,
        rolling_max,
    )
].copy()

rolling_ids = set(
    rolling_crime_compare[CRIME_KEY]
    .dropna()
    .astype(str)
)

full_ids = set(
    full_crime_compare[CRIME_KEY]
    .dropna()
    .astype(str)
)

missing_ids = full_ids - rolling_ids

missing_crimes = full_crime_compare[
    full_crime_compare[CRIME_KEY]
    .astype(str)
    .isin(missing_ids)
].copy()

print("Rolling unique offenses:", len(rolling_ids))
print("Full unique offenses:", len(full_ids))
print("Missing from rolling:", len(missing_ids))


Rolling unique offenses: 156694
Full unique offenses: 158120
Missing from rolling: 1430


### Observed result

The stale rolling snapshot was missing **1,103 offenses** that were present in the current full source over the same `offense_date` window.

This established that the discrepancy was real and large enough to investigate at the record level.


## 4. Determine how much of the discrepancy is explained by later reporting

The rolling snapshot's latest `report_date_time` is used as the cutoff representing the newest report timestamp captured in that snapshot.

A missing offense whose `report_date_time` is later than this cutoff could not have been captured by that snapshot yet, so it is categorized as **explained by later reporting**.


In [8]:
rolling_report_cutoff = rolling_crime[CRIME_REFRESH_TIME].max()

missing_crimes["reported_after_rolling_cutoff"] = (
    missing_crimes[CRIME_REFRESH_TIME] > rolling_report_cutoff
)

later_reported_count = int(
    missing_crimes["reported_after_rolling_cutoff"].sum()
)

not_later_reported = missing_crimes[
    ~missing_crimes["reported_after_rolling_cutoff"]
].copy()

pct_explained = (
    later_reported_count / len(missing_crimes) * 100
    if len(missing_crimes)
    else 0.0
)

print("Rolling report cutoff:", rolling_report_cutoff)
print("Total missing offenses:", len(missing_crimes))
print("Missing offenses reported after rolling cutoff:", later_reported_count)
print("Percent explained by later reporting:", round(pct_explained, 2))
print(
    "Missing offenses not explained by later reporting:",
    len(not_later_reported),
)


Rolling report cutoff: 2026-09-21 23:58:02
Total missing offenses: 1430
Missing offenses reported after rolling cutoff: 1257
Percent explained by later reporting: 87.9
Missing offenses not explained by later reporting: 173


### Observed result

- **1,103** offenses were missing from the rolling snapshot.
- **959** had `report_date_time` values later than the rolling snapshot cutoff.
- **86.94%** of the original discrepancy was therefore explained by later reporting.
- **144** missing records remained unexplained by that mechanism.

The remaining 144 records were the important cases for evaluating the incremental strategy itself.


## 5. Test whether the unexplained records fall inside the 30-day overlap

The production incremental strategy queries recent records using `report_date_time` with a 30-day overlap. Records inside that overlap should be eligible to be fetched on the next rolling refresh.


In [9]:
overlap_start = (
    rolling_report_cutoff.normalize()
    - pd.Timedelta(days=OVERLAP_DAYS)
)

not_later_reported["inside_30_day_overlap"] = (
    not_later_reported[CRIME_REFRESH_TIME] >= overlap_start
)

overlap_counts = (
    not_later_reported["inside_30_day_overlap"]
    .value_counts()
)

print("Overlap start:", overlap_start)
print(overlap_counts)


Overlap start: 2026-08-22 00:00:00
inside_30_day_overlap
True     151
False     22
Name: count, dtype: int64


### Observed result

Among the 144 missing records not explained by later reporting:

- **125** were inside the 30-day overlap.
- **19** were outside the overlap.

The 125 in-overlap records should be retrievable by the rolling query, so the next test checked that directly.


## 6. Verify that a fresh rolling-style query returns the 125 in-overlap records

This reproduces the production incremental query shape:

- `date_column="report_date_time"`
- start date = latest rolling `report_date_time` date minus 30 days
- full pagination


In [10]:
rolling_fetch_start = (
    rolling_report_cutoff.date()
    - timedelta(days=OVERLAP_DAYS)
).isoformat()

incremental_control = load_crime_dataset(
    start_date=rolling_fetch_start,
    page_size=5000,
    max_pages=None,
    date_column=CRIME_REFRESH_TIME,
)

incremental_control_ids = set(
    incremental_control[CRIME_KEY]
    .dropna()
    .astype(str)
)

inside_overlap_ids = set(
    not_later_reported.loc[
        not_later_reported["inside_30_day_overlap"],
        CRIME_KEY,
    ]
    .astype(str)
)

found_now = inside_overlap_ids & incremental_control_ids
still_not_returned = inside_overlap_ids - incremental_control_ids

print("Rolling-style fetch start:", rolling_fetch_start)
print("Expected missing IDs inside overlap:", len(inside_overlap_ids))
print("Returned by rolling-style query now:", len(found_now))
print("Still not returned:", len(still_not_returned))


Rolling-style fetch start: 2026-08-22
Expected missing IDs inside overlap: 151
Returned by rolling-style query now: 151
Still not returned: 0


### Observed result

The fresh rolling-style query returned **125 of 125** in-overlap missing IDs.

This ruled out the present-day `report_date_time` query definition as the cause of those missing records. The records were retrievable through the intended incremental query.


## 7. Simulate the production incremental merge

The next test applies the production rolling strategy in memory:

1. Start with the stale rolling snapshot.
2. Fetch the current 30-day `report_date_time` overlap.
3. Concatenate old and new records.
4. Deduplicate by `offense_id`, keeping the newest fetched version.
5. Recompute the 734-day retained window from the latest `offense_date`.

Nothing is written to disk.


In [11]:
sim_existing = rolling_crime.copy()
sim_new = incremental_control.copy()

for frame in [sim_existing, sim_new]:
    frame[CRIME_EVENT_TIME] = pd.to_datetime(
        frame[CRIME_EVENT_TIME],
        errors="coerce",
    )
    frame[CRIME_REFRESH_TIME] = pd.to_datetime(
        frame[CRIME_REFRESH_TIME],
        errors="coerce",
    )

sim_combined = pd.concat(
    [sim_existing, sim_new],
    ignore_index=True,
)

sim_combined = sim_combined.drop_duplicates(
    subset=[CRIME_KEY],
    keep="last",
)

sim_latest = sim_combined[CRIME_EVENT_TIME].max()
sim_cutoff = sim_latest - pd.Timedelta(days=ROLLING_WINDOW_DAYS)

sim_combined = sim_combined[
    sim_combined[CRIME_EVENT_TIME] >= sim_cutoff
].copy()

print("Simulated latest offense_date:", sim_latest)
print("Simulated rolling cutoff:", sim_cutoff)
print("Simulated rolling rows:", len(sim_combined))


Simulated latest offense_date: 2026-09-30 21:30:00
Simulated rolling cutoff: 2024-09-26 21:30:00
Simulated rolling rows: 157072


## 8. Compare the simulated rolling snapshot with the full source over identical bounds

This alignment is essential. Once the simulated refresh advances the latest `offense_date`, the 734-day cutoff also advances. Both data sets must therefore be restricted to the exact same `[sim_cutoff, sim_latest]` interval before comparing IDs.


In [12]:
full_control = full_crime[
    full_crime[CRIME_EVENT_TIME].between(
        sim_cutoff,
        sim_latest,
    )
].copy()

sim_control = sim_combined[
    sim_combined[CRIME_EVENT_TIME].between(
        sim_cutoff,
        sim_latest,
    )
].copy()

full_control_ids = set(
    full_control[CRIME_KEY]
    .dropna()
    .astype(str)
)

sim_control_ids = set(
    sim_control[CRIME_KEY]
    .dropna()
    .astype(str)
)

missing_after_refresh = full_control_ids - sim_control_ids
extra_after_refresh = sim_control_ids - full_control_ids

print("Comparison window:")
print("Start:", sim_cutoff)
print("End:  ", sim_latest)

print("\nFull unique offenses:", len(full_control_ids))
print("Simulated rolling unique offenses:", len(sim_control_ids))
print("\nMissing after simulated refresh:", len(missing_after_refresh))
print("Extra after simulated refresh:", len(extra_after_refresh))


Comparison window:
Start: 2024-09-26 21:30:00
End:   2026-09-30 21:30:00

Full unique offenses: 157090
Simulated rolling unique offenses: 157072

Missing after simulated refresh: 22
Extra after simulated refresh: 4


### Observed result

The aligned comparison produced:

- Comparison window: **2024-09-24 20:26:00 → 2026-09-28 20:26:00**
- Current full source: **156,934 unique offenses**
- Simulated rolling snapshot: **156,919 unique offenses**
- Missing after simulated refresh: **19**
- Extra after simulated refresh: **4**

The large 1,103-record discrepancy therefore collapsed to only 23 residual ID differences after a current incremental refresh was simulated.


## 9. Classify the 19 residual missing records

The residual missing records are checked against the same incremental fetch boundary. If all are older than that boundary, they demonstrate the overlap-window blind spot directly.


In [13]:
missing_residual = full_control[
    full_control[CRIME_KEY]
    .astype(str)
    .isin(missing_after_refresh)
].copy()

fetch_start_timestamp = pd.Timestamp(rolling_fetch_start)

missing_residual["inside_incremental_fetch_window"] = (
    missing_residual[CRIME_REFRESH_TIME]
    >= fetch_start_timestamp
)

print(
    missing_residual["inside_incremental_fetch_window"]
    .value_counts()
)

display(
    missing_residual[
        [
            CRIME_KEY,
            CRIME_EVENT_TIME,
            CRIME_REFRESH_TIME,
            "report_number",
            "offense_category",
            "offense_sub_category",
        ]
    ]
    .sort_values(CRIME_REFRESH_TIME)
)


inside_incremental_fetch_window
False    22
Name: count, dtype: int64


,offense_id,offense_date,report_date_time,report_number,offense_category,offense_sub_category
131254,73245471321,2025-01-15 06:05:00,2025-01-15 07:34:57,2025-013241,all other,assault offenses
124213,62859441899,2025-02-20 12:00:00,2025-02-20 16:51:42,2025-047367,property crime,motor vehicle theft
105712,64094797193,2025-05-22 11:00:00,2025-05-22 14:56:32,2025-138142,property crime,larceny-theft
104760,64141176153,2025-05-26 17:48:00,2025-05-26 19:48:32,2025-142668,all other,assault offenses
91187,65447871758,2025-07-29 00:00:00,2025-08-13 17:40:09,2025-232378,all other,999
73396,73361150674,2025-10-15 08:00:00,2025-10-16 19:32:21,2025-302368,all other,"property offenses (includes stolen, destruction)"
53364,68417281319,2026-01-14 13:20:00,2026-02-04 17:24:55,2026-034177,all other,extortion/fraud/forgery/bribery (includes bad ...
47242,68551896061,2026-02-13 16:00:00,2026-02-13 20:16:27,2026-043093,all other,extortion/fraud/forgery/bribery (includes bad ...
66574,72591924528,2025-11-14 00:00:00,2026-03-09 09:19:17,2026-028763,all other,999
66573,72591863682,2025-11-14 00:00:00,2026-03-09 09:19:17,2026-028763,all other,pornography


### Observed result

All **19 of 19** residual missing records were **outside** the 30-day incremental fetch window.

This confirms a structural limitation of the rolling strategy: records that become newly visible or change after their `report_date_time` has aged beyond the overlap cannot be recovered by later incremental refreshes.


## 10. Classify the 4 residual extra records

An extra rolling ID may represent either:

- a record whose fields moved it outside the comparison window, or
- a record that no longer exists in the current source.

The current full pull is searched without the aligned date-window restriction to distinguish those cases.


In [14]:
extra_residual = sim_control[
    sim_control[CRIME_KEY]
    .astype(str)
    .isin(extra_after_refresh)
].copy()

extra_ids = set(extra_after_refresh)

extra_in_full_anywhere = full_crime[
    full_crime[CRIME_KEY]
    .astype(str)
    .isin(extra_ids)
].copy()

ids_still_present = set(
    extra_in_full_anywhere[CRIME_KEY]
    .dropna()
    .astype(str)
)

ids_absent_from_current_source = extra_ids - ids_still_present

print("Extra IDs:", len(extra_ids))
print(
    "Extra IDs still present anywhere in current full pull:",
    len(ids_still_present),
)
print("\nIDs absent from current source entirely:")
print(sorted(ids_absent_from_current_source))

display(
    extra_residual[
        [
            CRIME_KEY,
            CRIME_EVENT_TIME,
            CRIME_REFRESH_TIME,
            "report_number",
            "offense_category",
            "offense_sub_category",
        ]
    ]
    .sort_values(CRIME_REFRESH_TIME)
)


Extra IDs: 4
Extra IDs still present anywhere in current full pull: 0

IDs absent from current source entirely:
['72513345011', '72513757285', '73063212435', '73235511023']


,offense_id,offense_date,report_date_time,report_number,offense_category,offense_sub_category
153336,72513757285,2026-08-29 19:28:00,2026-08-29 21:46:23,2026-257025,property crime,larceny-theft
153339,72513345011,2026-08-29 19:28:00,2026-08-29 21:46:23,2026-257025,violent crime,robbery
155382,73063212435,2026-09-11 08:41:00,2026-09-11 11:27:41,2026-270868,all other,unknown
155972,73235511023,2026-09-15 20:05:00,2026-09-15 22:34:54,2026-275787,all other,unknown


### Observed result

All **4** residual extra IDs were absent from the current full source entirely:

- `72513345011`
- `72513757285`
- `73063212435`
- `73235511023`

This is consistent with upstream record deletion/removal after those records had already been persisted in the rolling snapshot.

Because the rolling merge only adds/replaces keyed records, it has no mechanism to remove an upstream-deleted record until that record eventually ages out of the retained window.


# Conclusions

The QA established two distinct rolling-refresh failure modes.

## 1. Changes outside the overlap window

After simulating a fresh incremental refresh, all **19** records still missing from the rolling snapshot had `report_date_time` values outside the 30-day overlap.

A fixed overlap therefore cannot guarantee that the rolling snapshot eventually converges to the authoritative source.

## 2. Upstream deletions

All **4** records that existed only in the simulated rolling snapshot were no longer present anywhere in the current full source.

An append/merge/deduplicate refresh does not discover deletions for historical records that are never re-fetched.

## Operational decision

The current data volume is small enough that correctness is prioritized over minimizing source reads.

For the public dashboard, the preferred strategy is therefore:

1. perform a **full refresh of the retained source window daily**;
2. validate the completed pull before publishing it;
3. only replace the production snapshot after validation succeeds;
4. retain the previous known-good snapshot if the refresh fails.

This strategy directly reconciles current source additions, edits, and deletions instead of relying on a finite overlap window.

## Important scope note

These experiments specifically establish the behavior of the **crime data refresh**. They should not be interpreted as independent evidence about other dashboard sources unless equivalent QA is performed for those data streams.


# Full-refresh query-field decision: `offense_date` vs `report_date_time`

The rolling refresh used `report_date_time` because it needed a way to discover recently reported records, including offenses that occurred earlier.

Under a daily full-refresh strategy, that requirement changes. The refresh reconstructs the entire retained source window on every run.

The relevant question is therefore:

> Which query field produces the most complete representation of offenses whose `offense_date` falls inside the dashboard's retained 734-day window?

This experiment compares full source pulls using:

1. `offense_date >= target_start`
2. `report_date_time >= target_start`

Both results are then restricted to the **same `offense_date` window** before comparing unique offense IDs.

In [16]:
# ---------------------------------------------------------
# Define the dashboard's current retained offense-date window
# ---------------------------------------------------------

latest_offense_date = full_crime[CRIME_EVENT_TIME].max()

target_start = (
    latest_offense_date
    - pd.Timedelta(days=ROLLING_WINDOW_DAYS)
)

target_end = latest_offense_date

print("Target offense-date window:")
print("Start:", target_start)
print("End:  ", target_end)

Target offense-date window:
Start: 2024-09-26 21:30:00
End:   2026-09-30 21:30:00


In [17]:
# ---------------------------------------------------------
# Compare full pulls using the two candidate query fields
# ---------------------------------------------------------

query_start = target_start.date().isoformat()

full_by_offense_date = load_crime_dataset(
    start_date=query_start,
    page_size=5000,
    max_pages=None,
    date_column=CRIME_EVENT_TIME,
)

full_by_report_date = load_crime_dataset(
    start_date=query_start,
    page_size=5000,
    max_pages=None,
    date_column=CRIME_REFRESH_TIME,
)

for frame in [full_by_offense_date, full_by_report_date]:
    frame[CRIME_EVENT_TIME] = pd.to_datetime(
        frame[CRIME_EVENT_TIME],
        errors="coerce",
    )
    frame[CRIME_REFRESH_TIME] = pd.to_datetime(
        frame[CRIME_REFRESH_TIME],
        errors="coerce",
    )

print("Rows fetched using offense_date:", len(full_by_offense_date))
print("Rows fetched using report_date_time:", len(full_by_report_date))

Rows fetched using offense_date: 157367
Rows fetched using report_date_time: 160372


The raw row counts are not directly comparable because a `report_date_time` query can retrieve records whose offenses occurred before the dashboard window.

Both pulls must therefore be filtered to the exact same `offense_date` interval.

In [18]:
# ---------------------------------------------------------
# Restrict both pulls to identical dashboard semantics
# ---------------------------------------------------------

offense_query_control = full_by_offense_date[
    full_by_offense_date[CRIME_EVENT_TIME].between(
        target_start,
        target_end,
    )
].copy()

report_query_control = full_by_report_date[
    full_by_report_date[CRIME_EVENT_TIME].between(
        target_start,
        target_end,
    )
].copy()

offense_query_ids = set(
    offense_query_control[CRIME_KEY]
    .dropna()
    .astype(str)
)

report_query_ids = set(
    report_query_control[CRIME_KEY]
    .dropna()
    .astype(str)
)

only_offense_query = offense_query_ids - report_query_ids
only_report_query = report_query_ids - offense_query_ids

print("Unique offenses via offense_date query:",
      len(offense_query_ids))

print("Unique offenses via report_date_time query:",
      len(report_query_ids))

print()
print("Present only in offense_date pull:",
      len(only_offense_query))

print("Present only in report_date_time pull:",
      len(only_report_query))

Unique offenses via offense_date query: 157090
Unique offenses via report_date_time query: 157084

Present only in offense_date pull: 6
Present only in report_date_time pull: 0


In [20]:
# ---------------------------------------------------------
# Inspect target-window offenses missed by report-date query
# ---------------------------------------------------------

missed_by_report_query = offense_query_control[
    offense_query_control[CRIME_KEY]
    .astype(str)
    .isin(only_offense_query)
].copy()

print(
    "Dashboard-window offenses missed by "
    "report_date_time query:",
    len(missed_by_report_query)
)

if not missed_by_report_query.empty:
    display(
        missed_by_report_query[
            [
                CRIME_KEY,
                CRIME_EVENT_TIME,
                CRIME_REFRESH_TIME,
                "report_number",
                "offense_category",
                "offense_sub_category",
            ]
        ]
        .sort_values(CRIME_EVENT_TIME)
    )

Dashboard-window offenses missed by report_date_time query: 6


,offense_id,offense_date,report_date_time,report_number,offense_category,offense_sub_category
133648,53660413033,2025-01-04 00:00:00,2024-01-06 12:27:17,2024-005087,property crime,burglary
124473,54514088701,2025-02-19 00:00:00,2024-02-28 15:37:20,2024-048871,property crime,larceny-theft
102738,65267726062,2025-06-05 00:00:00,2024-07-30 10:07:38,2024-211466,all other,extortion/fraud/forgery/bribery (includes bad ...
47164,68608437628,2026-02-14 00:00:00,2023-12-18 03:50:23,2023-362475,all other,narcotic violations (includes drug equip.)
41002,69884650826,2026-03-17 20:50:00,2024-03-18 01:02:12,2024-073840,property crime,arson
10243,72091781866,2026-08-09 02:25:00,2023-11-09 06:40:36,2023-324168,property crime,burglary


In [21]:
# ---------------------------------------------------------
# Check offense/report temporal ordering
# ---------------------------------------------------------

temporal_control = offense_query_control[
    offense_query_control[CRIME_EVENT_TIME].notna()
    & offense_query_control[CRIME_REFRESH_TIME].notna()
].copy()

temporal_control["report_minus_offense"] = (
    temporal_control[CRIME_REFRESH_TIME]
    - temporal_control[CRIME_EVENT_TIME]
)

report_before_offense = temporal_control[
    temporal_control["report_minus_offense"]
    < pd.Timedelta(0)
].copy()

print("Records evaluated:", len(temporal_control))

print(
    "Records with report_date_time before offense_date:",
    len(report_before_offense),
)

print(
    "Percent:",
    round(
        len(report_before_offense)
        / len(temporal_control)
        * 100,
        4,
    ),
)

Records evaluated: 157096
Records with report_date_time before offense_date: 418
Percent: 0.2661


In [22]:
# ---------------------------------------------------------
# Measure records fetched unnecessarily by report_date_time
# ---------------------------------------------------------

report_query_outside_window = full_by_report_date[
    ~full_by_report_date[CRIME_EVENT_TIME].between(
        target_start,
        target_end,
    )
].copy()

print(
    "Rows fetched by report_date_time but outside "
    "the dashboard offense-date window:",
    len(report_query_outside_window),
)

if len(full_by_report_date):
    print(
        "Percent of report-date pull outside dashboard window:",
        round(
            len(report_query_outside_window)
            / len(full_by_report_date)
            * 100,
            2,
        ),
    )

Rows fetched by report_date_time but outside the dashboard offense-date window: 3282
Percent of report-date pull outside dashboard window: 2.05


In [23]:
# ---------------------------------------------------------
# Compare daily dashboard counts produced by each query
# ---------------------------------------------------------

offense_daily = (
    offense_query_control
    .assign(
        date=offense_query_control[
            CRIME_EVENT_TIME
        ].dt.normalize()
    )
    .groupby("date")[CRIME_KEY]
    .nunique()
    .rename("offense_date_query")
)

report_daily = (
    report_query_control
    .assign(
        date=report_query_control[
            CRIME_EVENT_TIME
        ].dt.normalize()
    )
    .groupby("date")[CRIME_KEY]
    .nunique()
    .rename("report_date_query")
)

daily_query_comparison = pd.concat(
    [offense_daily, report_daily],
    axis=1,
).fillna(0)

daily_query_comparison["difference"] = (
    daily_query_comparison["report_date_query"]
    - daily_query_comparison["offense_date_query"]
)

print(
    "Days with different unique-offense totals:",
    (daily_query_comparison["difference"] != 0).sum(),
)

print(
    "Maximum absolute daily difference:",
    daily_query_comparison["difference"].abs().max(),
)

display(
    daily_query_comparison[
        daily_query_comparison["difference"] != 0
    ]
)

Days with different unique-offense totals: 6
Maximum absolute daily difference: 1


,offense_date_query,report_date_query,difference
date,,,
2025-01-04,204,203,-1
2025-02-19,176,175,-1
2025-06-05,242,241,-1
2026-02-14,211,210,-1
2026-03-17,206,205,-1
2026-08-09,209,208,-1
